> ## ⚠️  Superseded — single-run draft, retained as a record
>
> **This notebook will not run.** It opens `processed_data/all_data.pkl`, the earlier
> 53-feature preprocessing artefact that was replaced by `canonical_data.pkl` when the
> feature set was corrected. Nothing in this folder regenerates it, so the first cell raises
> `FileNotFoundError`.
>
> **No result reported in the thesis comes from this notebook.** Every reported figure is
> produced by the canonical path `00`/`01` → `08`–`15`, which trains over thirty fixed seeds
> (`SEEDS = [42 + 7*i for i in range(30)]`). Model definitions — `TemporalAttention`,
> `build_lstm_transformer`, `build_standalone_lstm` — live in `canonical_pipeline.py`.
>
> It is kept because its saved outputs are the record of the single-run baseline stage the
> project passed through. See README §5.
>
> The ILT architecture shown here is the same one defined in `canonical_pipeline.py`; the
> thirty-seed training that produced Table 5.2 is in `08_multi_horizon_comparison_30run_RESULT_REVIEWED.ipynb`.

# 06 — Train Interpretable LSTM-Transformer (Proposed)

The hybrid architecture that combines:
- **LSTM encoder**: temporal memory for sequential dengue patterns
- **Multi-head self-attention**: interpretable weighting of which past time steps matter most
- **Feature branch**: direct meteorological input
- **Residual connection + LayerNorm**: stable training

This is the core contribution of the thesis (Sections 2.3.2, 2.3.3).

In [1]:
import pickle, numpy as np, warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
np.random.seed(42); tf.random.set_seed(42)

with open('processed_data/all_data.pkl', 'rb') as f:
    data = pickle.load(f)
X_tr_seq_, X_tr_feat_, y_tr_seq_ = data['X_tr_seq_'], data['X_tr_feat_'], data['y_tr_seq_']
X_val_seq, X_val_feat, y_val_seq = data['X_val_seq'], data['X_val_feat'], data['y_val_seq']
X_te_seq, X_te_feat = data['X_te_seq'], data['X_te_feat']
y_te_seq_raw = data['y_te_seq_raw']
target_scaler = data['target_scaler']
SEQ_LEN = data['SEQ_LEN']
n_features = X_tr_seq_.shape[2]
print(f"Train: {len(X_tr_seq_)}, Val: {len(X_val_seq)}, Test: {len(X_te_seq)}")

Train: 2431, Val: 429, Test: 695


## AttentionExtractor Layer

In [2]:
class AttentionExtractor(layers.Layer):
    """Wraps MultiHeadAttention and stores attention weights for extraction."""
    def __init__(self, num_heads, key_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim, dropout=dropout)
        self.attention_weights = None
    def call(self, query, value, training=False):
        output, weights = self.mha(query, value, return_attention_scores=True, training=training)
        self.attention_weights = weights
        return output
    def get_config(self):
        config = super().get_config()
        config.update({'num_heads': self.mha.num_heads, 'key_dim': self.mha._key_dim, 'dropout': self.mha._dropout})
        return config
print("✅ AttentionExtractor defined")

✅ AttentionExtractor defined


## Build Model

In [3]:
si = Input(shape=(SEQ_LEN, n_features), name='sequence_input')
fi = Input(shape=(n_features,), name='feature_input')

# LSTM encoder — temporal memory
x = layers.LSTM(16, return_sequences=True, dropout=0.2, name='lstm_encoder')(si)
x = layers.LayerNormalization(name='lstm_ln')(x)

# Self-attention on LSTM hidden states — interpretability
attn_layer = AttentionExtractor(num_heads=2, key_dim=8, dropout=0.1, name='attention_extractor')
attn_out = attn_layer(x, x)
attn_out = layers.LayerNormalization(name='attn_ln')(attn_out + x)  # residual
attn_pooled = layers.GlobalAveragePooling1D(name='attn_pool')(attn_out)

# Feature branch
feat = layers.Dense(16, activation='relu', name='feat_d')(fi)
feat = layers.Dropout(0.2, name='feat_do')(feat)

# Prediction head
combined = layers.Concatenate(name='merge')([attn_pooled, feat])
h = layers.Dense(16, activation='relu', name='hd1')(combined)
h = layers.Dropout(0.2, name='do1')(h)
o = layers.Dense(1, activation='sigmoid', name='forecast')(h)

model = Model(inputs=[si, fi], outputs=o, name='InterpretableLSTM_Transformer')
model.attn_layer = attn_layer
model.compile(optimizer=Adam(learning_rate=0.001, clipnorm=1.0), loss='mse', metrics=['mae'])

print(f"LSTM-Transformer: {model.count_params():,} params")
print(f"Params/sample: {model.count_params()/len(X_tr_seq_):.2f}")
model.summary()


LSTM-Transformer: 7,041 params
Params/sample: 2.90


Model: "InterpretableLSTM_Transformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 28, 53)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_encoder (LSTM) │ (None, 28, 16)    │      4,480 │ sequence_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_ln             │ (None, 28, 16)    │         32 │ lstm_encoder[0][… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_extractor │ (None, 28, 16)    │      1,088 │ lstm_ln[0][0],    │
│ (AttentionExtracto… │                   │            │ lstm_ln[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 28, 16)    │          0 │ attention_extrac… │
│                     │                   │            │ lstm_ln[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_input       │ (None, 53)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_ln             │ (None, 28, 16)    │         32 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feat_d (Dense)      │ (None, 16)        │        864 │ feature_input[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_pool           │ (None, 16)        │          0 │ attn_ln[0][0]     │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feat_do (Dropout)   │ (None, 16)        │          0 │ feat_d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ merge (Concatenate) │ (None, 32)        │          0 │ attn_pool[0][0],  │
│                     │                   │            │ feat_do[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hd1 (Dense)         │ (None, 16)        │        528 │ merge[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ do1 (Dropout)       │ (None, 16)        │          0 │ hd1[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ forecast (Dense)    │ (None, 1)         │         17 │ do1[0][0]         │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 7,041 (27.50 KB)

 Trainable params: 7,041 (27.50 KB)

 Non-trainable params: 0 (0.00 B)

## Train

In [4]:
print("Training LSTM-Transformer...")
_ = model.predict([X_tr_seq_[:1], X_tr_feat_[:1]], verbose=0)  # warmup to compile graph

history = model.fit(
    [X_tr_seq_, X_tr_feat_], y_tr_seq_,
    validation_data=([X_val_seq, X_val_feat], y_val_seq),
    epochs=100, batch_size=32, verbose=0,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-7, verbose=0),
    ])

pred_s = model.predict([X_te_seq, X_te_feat], verbose=0).flatten()
pred = np.maximum(target_scaler.inverse_transform(pred_s.reshape(-1,1)).flatten(), 0)
rmse = np.sqrt(mean_squared_error(y_te_seq_raw, pred))
mae = mean_absolute_error(y_te_seq_raw, pred)
r2 = r2_score(y_te_seq_raw, pred)

best = np.argmin(history.history['val_loss']) + 1
print(f"\n✅ LSTM-Transformer — Best epoch: {best}/{len(history.history['val_loss'])}")
print(f"   RMSE: {rmse:.2f}, MAE: {mae:.2f}, R²: {r2:.4f}")

Training LSTM-Transformer...
Epoch 23: early stopping
Restoring model weights from the end of the best epoch: 3.

✅ LSTM-Transformer — Best epoch: 3/23
   RMSE: 24.14, MAE: 13.50, R²: 0.8178


## Extract Attention Weights

In [5]:
# Extract attention weights for interpretability analysis
# model.predict() returns SymbolicTensor for custom layers.
# Direct model call model([...], training=False) uses eager mode → EagerTensor.

def extract_attention_weights(model, X_seq, X_feat, batch_size=64):
    all_weights = []
    for i in range(0, len(X_seq), batch_size):
        batch_seq = tf.constant(X_seq[i:i+batch_size], dtype=tf.float32)
        batch_feat = tf.constant(X_feat[i:i+batch_size], dtype=tf.float32)
        _ = model([batch_seq, batch_feat], training=False)  # eager call
        all_weights.append(model.attn_layer.attention_weights.numpy())
    return np.concatenate(all_weights, axis=0)

attn_weights = extract_attention_weights(model, X_te_seq, X_te_feat)
print(f"Attention weights shape: {attn_weights.shape}")
print(f"  = (test_samples, heads, query_steps, key_steps)")

# Feature importance via attention
temporal_importance = attn_weights.mean(axis=(0, 1, 2))
feature_importance = np.zeros(n_features)
for t in range(SEQ_LEN):
    feature_importance += temporal_importance[t] * np.abs(X_te_seq[:, t, :]).mean(axis=0)
feature_importance /= feature_importance.max()
print(f"Feature importance computed for {n_features} features")

Attention weights shape: (695, 2, 28, 28)
  = (test_samples, heads, query_steps, key_steps)
Feature importance computed for 53 features


In [6]:
# Save everything
results = {'model': 'LSTM-Transformer', 'RMSE': rmse, 'MAE': mae, 'R2': r2,
           'predictions': pred, 'history': history.history}
with open('metrics/lstm_transformer_results.pkl', 'wb') as f:
    pickle.dump(results, f)
model.save('models/lstm_transformer.keras')
np.save('attention/weights.npy', attn_weights)
np.save('attention/temporal_importance.npy', temporal_importance)
np.save('attention/feature_importance.npy', feature_importance)
print("✅ Saved model, metrics, and attention weights")

✅ Saved model, metrics, and attention weights
